# RLHF Part 2: PPO Fine-Tuning

With a trained reward model, the next step is using Proximal Policy Optimization (PPO) to fine-tune the language model to maximize reward while staying close to the original policy. This is the core RLHF loop that produced InstructGPT and ChatGPT.

## The RLHF Objective

The PPO objective for language model alignment maximizes expected reward while penalizing divergence from the reference model:

```
J(θ) = E[r(x, y)] - β * KL(π_θ(y|x) || π_ref(y|x))
```

Where:
- `r(x, y)` is the reward model score for response y to prompt x
- `π_θ` is the policy being optimized
- `π_ref` is the frozen reference model (the SFT model before RL training)
- `β` is the KL coefficient controlling how much the policy can deviate

The KL penalty is critical: without it, the policy quickly collapses to reward hacking — generating nonsense that achieves high reward but has completely degenerated quality.

## PPO Instability History

PPO fine-tuning of large language models proved difficult in practice:

**Reward hacking**: the policy discovers ways to maximize the reward model's score that do not correspond to genuine quality. Common patterns: extreme verbosity, specific phrases that the RM rated highly, repetitive safe-sounding language.

**KL explosion**: if the KL coefficient is too low, the policy diverges rapidly. If it is too high, learning is suppressed. Tuning β is critical.

**Value function instability**: PPO requires a value function (critic) that estimates expected return from a given state. Training the critic alongside the policy in the same forward pass creates optimization interference.

**Mode collapse**: the policy stops exploring and commits to a narrow set of high-reward outputs.

These instabilities motivated the search for simpler alignment methods, leading directly to DPO.

In [ ]:
import math
import random
from dataclasses import dataclass

@dataclass
class PPOConfig:
    kl_coef: float = 0.1
    clip_eps: float = 0.2
    lr: float = 1e-5
    n_steps: int = 100

@dataclass
class RLHFStep:
    prompt: str
    response: str
    reward: float
    kl_penalty: float
    adjusted_reward: float
    policy_update: float

def compute_kl_penalty(log_prob_policy: float, log_prob_ref: float) -> float:
    # KL(policy || reference) approximated as log(policy/reference)
    return log_prob_policy - log_prob_ref

def ppo_clip_objective(ratio: float, advantage: float, eps: float = 0.2) -> float:
    clipped = max(1 - eps, min(1 + eps, ratio))
    return min(ratio * advantage, clipped * advantage)

class SimplePPOLoop:
    def __init__(self, config: PPOConfig, seed: int = 42):
        self.config = config
        self.rng = random.Random(seed)
        self.policy_quality = 5.0  # scalar proxy for model quality
        self.history = []

    def generate(self, prompt: str) -> tuple:
        # Simulate policy generating a response
        noise = self.rng.gauss(0, 0.5)
        response = f'Response at quality {self.policy_quality:.2f}'
        log_prob = -len(response) * 0.01 + noise  # simplified
        return response, log_prob

    def reward_model(self, prompt: str, response: str) -> float:
        # Reward increases with quality but has diminishing returns
        return math.tanh(self.policy_quality / 10) * 10 + self.rng.gauss(0, 0.3)

    def step(self, prompt: str, ref_log_prob: float) -> RLHFStep:
        response, policy_log_prob = self.generate(prompt)
        reward = self.reward_model(prompt, response)
        kl = compute_kl_penalty(policy_log_prob, ref_log_prob)
        adjusted = reward - self.config.kl_coef * kl
        # Update policy toward higher reward
        update = self.config.lr * adjusted
        self.policy_quality = min(10, self.policy_quality + update)
        return RLHFStep(prompt=prompt, response=response[:40],
                        reward=round(reward, 3), kl_penalty=round(kl, 3),
                        adjusted_reward=round(adjusted, 3), policy_update=round(update, 5))

config = PPOConfig(kl_coef=0.1, lr=0.05)
loop = SimplePPOLoop(config)
ref_log_prob = -2.0

print(f'PPO Training (kl_coef={config.kl_coef})')
print(f'{'Step':>5} {'Reward':>8} {'KL':>8} {'Adjusted':>10} {'Quality':>9}')
print('-' * 45)
for i in range(10):
    step = loop.step('What is gravity?', ref_log_prob)
    print(f'{i+1:>5} {step.reward:>8.3f} {step.kl_penalty:>8.3f} {step.adjusted_reward:>10.3f} {loop.policy_quality:>9.3f}')

## The KL Coefficient Tradeoff

The KL coefficient β is the most important RLHF hyperparameter:

- **β too low**: the policy drifts far from the reference model, enabling reward hacking. The model may learn to output gibberish that scores highly with the reward model.
- **β too high**: the policy cannot move far from the reference, suppressing alignment improvement.
- **Adaptive β**: some implementations dynamically adjust β based on the observed KL divergence — increasing it if KL grows too fast, decreasing it if KL is too low.

OpenAI's InstructGPT paper used β=0.01-0.1. The Anthropic RLHF papers use similar ranges. In practice, tuning β is dataset and model specific.

In [ ]:
def simulate_kl_tradeoff(kl_values: list, n_steps: int = 20) -> list:
    results = []
    for kl_coef in kl_values:
        rng = random.Random(42)
        quality = 5.0
        reward_hacking = 0.0  # proxy for degeneration
        for _ in range(n_steps):
            raw_reward = quality * 1.2 + reward_hacking + rng.gauss(0, 0.3)
            kl = abs(rng.gauss(0, 0.5)) * (10 / (kl_coef * 10 + 1))  # higher coef = less drift
            adjusted = raw_reward - kl_coef * kl
            quality = min(10, quality + 0.05 * adjusted)
            if kl_coef < 0.05:  # low KL -> reward hacking grows
                reward_hacking = min(5, reward_hacking + 0.02)
        results.append({
            'kl_coef': kl_coef,
            'final_quality': round(quality, 2),
            'reward_hacking': round(reward_hacking, 2),
        })
    return results

results = simulate_kl_tradeoff([0.001, 0.01, 0.05, 0.1, 0.5, 1.0])
print(f'{'kl_coef':>10} {'Final Quality':>14} {'Reward Hacking':>16}')
print('-' * 45)
for r in results:
    print(f"{r['kl_coef']:>10} {r['final_quality']:>14.2f} {r['reward_hacking']:>16.2f}")

## Why PPO Was Replaced

Despite producing strong results, PPO for LLM alignment has several practical drawbacks:
- Requires loading four models simultaneously: policy, reference, reward model, and value function
- Hyperparameter sensitivity: KL coefficient, PPO clip epsilon, and value loss coefficient all interact
- Sample efficiency: generates many low-reward responses that don't contribute useful training signal
- Engineering complexity: the PPO loop is significantly more complex than supervised fine-tuning

DPO (Direct Preference Optimization, Rafailov et al. 2023) eliminated the RL loop entirely, deriving an equivalent objective that can be trained with a standard supervised loss. This is covered in the next notebook.